# Rescue a broken report

*Python Foundations · Lesson 11 final exercise*

The marketing team wants the Black Friday campaigns' sessions by market and their revenue from November 21 to 30, 2024. December 1 and 2 go in December's report. Your colleague's notebook below has an answer under every cell, but it doesn't run from the top, and when it does, its numbers don't agree with each other. Rescue it: make it run, make it answer the team's question, and turn it into a report that the team can run again for another campaign by changing one cell.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Download [sessions_export_2024_11.csv](https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/sessions_export_2024_11.csv) and upload it to the `simmer-python` folder in your Drive.
2. Read your colleague's execution counts, then choose **Runtime → Restart session and run all**. Fix the notebook until every cell runs from the top.
3. Add a parameters cell above every other code cell: `CSV_PATH`, `START_DATE`, `END_DATE` and `CAMPAIGN_THEME` (`"blackfriday"`). No cell below it should repeat their values.
4. Make every result use the same rows, with these names:
   - `file_dates`: a sorted list of the dates in `rows`, each one once
   - `campaign_rows`: the rows whose campaign has the theme, from `START_DATE` to `END_DATE`
   - `market_sessions`: each market's sessions in `campaign_rows`
   - `campaign_revenue`: the total revenue of `campaign_rows`, as a float
   - `days`: the number of days from `START_DATE` to `END_DATE`, with both ends counted
5. Replace the last cell with one that creates `summary` and prints it, in the format from topic 11.2.: `From 2024-11-21 to 2024-11-30 (10 days), the blackfriday campaigns brought … sessions and $… in revenue.`
6. Replace your colleague's text cell with a title and a sentence that say what the report shows and how to run it for something else, and add a heading and a sentence above each part. No numbers in the text cells.
7. Choose **Runtime → Restart session and run all**, and read the check's output.

**Expected output**: the summary says 14,590 sessions and $43,928.81 in revenue. The **Check your work** cell runs last, with the others.

Black Friday 2024: 14,590 sessions, $56,050.04 revenue, 12 days.

In [1]:
import csv
from datetime import date

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
with open("/content/drive/MyDrive/simmer-python/sessions_export_2024_11.csv", newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))
len(rows)

2023

In [3]:
bf_rows = [row for row in rows if row["session_campaign"].endswith("_blackfriday") and "2024-11-21" <= row["date"] <= "2024-11-30"]
len(bf_rows)

180

In [4]:
market_sessions = {}
for row in bf_rows:
    market = row["session_campaign"].split("_")[1]
    market_sessions[market] = market_sessions.get(market, 0) + int(row["sessions"])
market_sessions

{'ca': 4175, 'us': 10415}

In [7]:
bf_revenue = 0
for row in rows:
    if row["session_campaign"].endswith("_blackfriday") and "2024-11-21" <= row["date"] <= "2024-12-02":
        bf_revenue += parse_revenue(row["revenue"])
bf_revenue

56050.03999999998

In [6]:
def parse_revenue(value):
    """Convert revenue text such as "$1,459.73" or "-$438.92" to a float."""
    return float(value.replace("$", "").replace(",", ""))

In [8]:
print((date(2024, 12, 2) - date(2024, 11, 21)).days + 1, sum(market_sessions.values()), round(bf_revenue, 2))

12 14590 56050.04


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import os
    import re
    import sys
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    clean = False

    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            clean = True
            results.append((True, "Every cell in this session ran once, with no errors."))

    parameters = {"START_DATE": "2024-11-21", "END_DATE": "2024-11-30", "CAMPAIGN_THEME": "blackfriday"}
    path = found.get("CSV_PATH")
    wrong = [f"{name} is {found.get(name)!r}" for name, value in parameters.items() if found.get(name) != value]
    if not isinstance(path, str):
        results.append((False, "CSV_PATH isn't defined. Add the parameters cell above every other code cell, and run it."))
    elif not path.replace("\\", "/").endswith("sessions_export_2024_11.csv"):
        results.append((False, f"CSV_PATH is {path!r}. The report uses November's export, sessions_export_2024_11.csv."))
    elif not os.path.exists(path):
        results.append((False, f"There's no file at {path!r}. Check that Drive is mounted and that the file is in the simmer-python folder, then copy its path from the Files panel."))
    elif found.get("END_DATE") == "2024-12-02":
        results.append((False, "END_DATE is \"2024-12-02\". The team wants November 21 to 30: December 1 and 2 go in December's report."))
    elif wrong:
        results.append((False, f"{wrong[0]}. The parameters should be START_DATE = \"2024-11-21\", END_DATE = \"2024-11-30\" and CAMPAIGN_THEME = \"blackfriday\", with the zeros."))
    else:
        results.append((True, "The parameters are November's export, November 21 to 30 and the Black Friday campaigns."))

    if history is not None:
        names = ["CSV_PATH", *parameters]
        cells = [source for source in history if re.search(r"^START_DATE\s*=", source, re.M)]
        hardcoded = ["blackfriday", "2024-11", "2024-12", "2024, 11", "2024, 12", "sessions_export_2024_11"]
        if not cells:
            results.append((False, "No cell that has run sets START_DATE. Add the parameters cell above every other code cell."))
        else:
            parameter_cell = cells[-1]
            missing = [name for name in names if not re.search(rf"^{name}\s*=", parameter_cell, re.M)]
            below = [(value, source) for source in history if source not in cells for value in hardcoded if value in source]
            if missing:
                results.append((False, f"The parameters cell doesn't set {', '.join(missing)}. Put all four parameters in the same cell."))
            elif clean and history[0] != parameter_cell:
                results.append((False, f"The parameters cell isn't the first code cell: the first one starts with {history[0].strip().splitlines()[0]!r}. Move the parameters cell to the top."))
            elif below:
                value, source = below[0]
                results.append((False, f"{value!r} is still hardcoded in the cell that starts with {source.strip().splitlines()[0]!r}. Use the parameter instead."))
            else:
                results.append((True, "The parameters are in one cell, and no other cell repeats them."))

    parse_revenue = found.get("parse_revenue")
    if not callable(parse_revenue):
        results.append((False, "parse_revenue() isn't defined. Move the cell that defines it above the cell that uses it."))
    else:
        try:
            parsed = [parse_revenue("$1,459.73"), parse_revenue("-$438.92")]
        except Exception as error:
            parsed = f"{type(error).__name__}: {error}"
        if parsed == [1459.73, -438.92]:
            results.append((True, "parse_revenue() converts the export's revenue text."))
        else:
            results.append((False, f"parse_revenue(\"$1,459.73\") and parse_revenue(\"-$438.92\") should give 1459.73 and -438.92, but they gave {parsed!r}. Keep your colleague's parse_revenue() as it was."))

    rows = found.get("rows")
    file_dates = found.get("file_dates")
    if not isinstance(rows, list):
        results.append((False, "rows isn't a list. Run the cell that reads the file."))
    elif len(rows) != 2023:
        results.append((False, f"rows has {len(rows)} rows, but sessions_export_2024_11.csv has 2,023."))
    elif not isinstance(file_dates, list):
        results.append((False, "file_dates should be a sorted list of the dates in rows, each one once: sorted({row[\"date\"] for row in rows})."))
    elif len(file_dates) == 2023:
        results.append((False, "file_dates has every row's date. Put them in a set first, so each date is there once."))
    elif file_dates != sorted(file_dates):
        results.append((False, "file_dates isn't sorted. Use sorted() on the set of dates."))
    elif len(file_dates) == 32 and file_dates[-2:] == ["2024-12-01", "2024-12-02"]:
        results.append((True, "file_dates has the file's 32 dates, including December 1 and 2."))
    else:
        results.append((False, f"file_dates has {len(file_dates)} dates, but the file has 32, from 2024-11-01 to 2024-12-02."))

    campaign_rows = found.get("campaign_rows")
    if not isinstance(campaign_rows, list):
        results.append((False, "campaign_rows isn't defined. Rename bf_rows to campaign_rows, in every cell that uses it."))
    elif len(campaign_rows) == 180:
        results.append((True, "campaign_rows has the 180 Black Friday rows from November 21 to 30."))
    elif len(campaign_rows) == 217:
        results.append((False, "campaign_rows has 217 rows: it includes December 1 and 2. Keep the rows where START_DATE <= row[\"date\"] <= END_DATE."))
    else:
        results.append((False, f"campaign_rows has {len(campaign_rows)} rows, but there are 180 Black Friday rows from November 21 to 30. Keep the rows whose campaign ends with \"_\" + CAMPAIGN_THEME, from START_DATE to END_DATE."))

    expected = {"ca": 4175, "us": 10415}
    sessions = found.get("market_sessions")
    if sessions == expected:
        results.append((True, "market_sessions has each market's sessions."))
    elif isinstance(sessions, dict) and all(sessions.get(market) == 2 * value for market, value in expected.items()):
        results.append((False, "market_sessions has twice the sessions: its loop ran twice. Start the dictionary in the loop's cell, then restart and run all."))
    elif sessions == {"ca": 5403, "us": 13592}:
        results.append((False, "market_sessions includes December 1 and 2. Total the sessions of campaign_rows, and check END_DATE."))
    else:
        results.append((False, f"market_sessions should be {expected}, but it is {sessions!r}. Total the sessions of campaign_rows by market."))

    revenue = found.get("campaign_revenue")
    if not isinstance(revenue, (int, float)):
        results.append((False, "campaign_revenue isn't defined. Rename bf_revenue to campaign_revenue, in every cell that uses it."))
    elif abs(revenue - 43928.81) < 0.01:
        results.append((True, "campaign_revenue is $43,928.81, from the same rows as the sessions."))
    elif abs(revenue - 56050.04) < 0.01:
        results.append((False, "campaign_revenue is $56,050.04, your colleague's number, which includes December 1 and 2. Total the revenue of campaign_rows with parse_revenue(), so it comes from the same rows as the sessions, and check END_DATE."))
    else:
        results.append((False, f"campaign_revenue should be about 43928.81, but it is {revenue!r}. Add parse_revenue(row[\"revenue\"]) for every row in campaign_rows."))

    days = found.get("days")
    if days == 10:
        results.append((True, "days is 10."))
    elif days == 12:
        results.append((False, "days is 12: it counts to December 2. Use the parameters: (date.fromisoformat(END_DATE) - date.fromisoformat(START_DATE)).days + 1."))
    elif days == 9:
        results.append((False, "days is 9: it doesn't count both ends. Add 1 to the difference's .days."))
    else:
        results.append((False, f"days should be 10, but it is {days!r}. Use (date.fromisoformat(END_DATE) - date.fromisoformat(START_DATE)).days + 1."))

    summary = found.get("summary")
    expected_summary = "From 2024-11-21 to 2024-11-30 (10 days), the blackfriday campaigns brought 14,590 sessions and $43,928.81 in revenue."
    if summary == expected_summary:
        results.append((True, "summary is the right sentence."))
    elif not isinstance(summary, str):
        results.append((False, "summary isn't defined. Create it in the last cell, with an f-string."))
    else:
        results.append((False, f"summary should be:\n   {expected_summary}\n   but it is:\n   {summary}"))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. The report is ready for the team, and for next year's Black Friday.")


check_your_work()

<details>
<summary><b>Show hint</b></summary>

Restart and run all first: the error tells you which cell is in the wrong place, as in topic 11.1. Then look for every date and every `blackfriday` in the code. The revenue cell has its own filter, over `rows`, with another end date: replace it with a loop over `campaign_rows`, so the sessions and the revenue come from the same rows. `file_dates`, the parameters cell, `days` and `summary` are built as in topic 11.2.

</details>

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to fix it: if the check passes, your way is right too. The fixed notebook, from the top:

Move `parse_revenue()` above the cells that use it. Then add the parameters cell, use one set of rows for the sessions, the revenue and the days, and replace the text cell and the last cell.

A text cell:

```markdown
# Campaign theme report

Sessions by market and revenue for one campaign theme over a date range, from the store's session export. To run it for another theme or another range, change the parameters in the first code cell, then choose **Runtime → Restart session and run all**.

## Parameters
```

```python
CSV_PATH = "/content/drive/MyDrive/simmer-python/sessions_export_2024_11.csv"
START_DATE = "2024-11-21"
END_DATE = "2024-11-30"
CAMPAIGN_THEME = "blackfriday"
```

A text cell:

```markdown
## Data

The export, and the dates it covers. Exports can include days from the next month, so the report keeps only the rows from START_DATE to END_DATE. Revenue is exported as text such as `"$1,459.73"`, so `parse_revenue()` converts it.
```

```python
import csv
from datetime import date

from google.colab import drive

drive.mount("/content/drive")
```

```python
def parse_revenue(value):
    """Convert revenue text such as "$1,459.73" or "-$438.92" to a float."""
    return float(value.replace("$", "").replace(",", ""))
```

```python
with open(CSV_PATH, newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))

file_dates = sorted({row["date"] for row in rows})
len(rows), file_dates[0], file_dates[-1]
```

```python
campaign_rows = [
    row for row in rows
    if row["session_campaign"].endswith("_" + CAMPAIGN_THEME) and START_DATE <= row["date"] <= END_DATE
]
len(campaign_rows)
```

A text cell:

```markdown
## Sessions by market

The market is the second part of the campaign name, as in `2024_us_email_blackfriday`.
```

```python
market_sessions = {}
for row in campaign_rows:
    market = row["session_campaign"].split("_")[1]
    market_sessions[market] = market_sessions.get(market, 0) + int(row["sessions"])
market_sessions
```

A text cell:

```markdown
## Revenue

Total revenue of the same rows, refunds included.
```

```python
campaign_revenue = 0
for row in campaign_rows:
    campaign_revenue += parse_revenue(row["revenue"])
campaign_revenue
```

A text cell:

```markdown
## Summary
```

```python
days = (date.fromisoformat(END_DATE) - date.fromisoformat(START_DATE)).days + 1
total_sessions = sum(market_sessions.values())
summary = f"From {START_DATE} to {END_DATE} ({days} days), the {CAMPAIGN_THEME} campaigns brought {total_sessions:,} sessions and ${campaign_revenue:,.2f} in revenue."
print(summary)
```

</details>